# Model Training

In [64]:
import importlib

import pytorch_lightning as pl
from torch.utils.data import DataLoader
from pytorch_lightning.loggers import TensorBoardLogger
from Config import constants
from Config import model_configs
import torch
import random
import numpy as np
import pandas as pd
from src.preprocessing.GTSRBDataset import GTSRBDataset
from src.preprocessing.transformations import get_transforms
from src.models.experiment_manager import get_experiment_setup
from src.models.model_template import TrafficSignClassifier
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

importlib.reload(constants)
importlib.reload(model_configs)

<module 'Config.model_configs' from 'F:\\Software\\Dataspell\\TrafficSigns\\Config\\model_configs.py'>

## Set Seed

In [65]:
torch.manual_seed(constants.SEED)
random.seed(constants.SEED)
np.random.seed(constants.SEED)

## GPU

In [66]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA GeForce RTX 3070


## Model Config

In [67]:
cfg_name = "exp004"

In [68]:
cfg = model_configs.CONFIGS[cfg_name]

IMAGE_SIZE = cfg["image_size"]
BATCH_SIZE = cfg["batch_size"]
lr = cfg["lr"]

## Train Test Split

In [69]:
train_df = pd.read_csv(constants.TRAIN_CSV_PATH)
val_df = pd.read_csv(constants.VAL_CSV_PATH)
test_df = pd.read_csv(constants.TEST_CSV_PATH)

In [70]:
train_ds = GTSRBDataset(train_df, transform=get_transforms(IMAGE_SIZE))
val_ds = GTSRBDataset(val_df, transform=get_transforms(IMAGE_SIZE, train=False))
test_ds = GTSRBDataset(val_df, transform=get_transforms(IMAGE_SIZE, train=False))

In [71]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

## Model

In [72]:
experiment_setup = get_experiment_setup(cfg, num_classes=constants.NUM_CLASSES)

In [73]:
model = experiment_setup["model"]
optimizer = experiment_setup["optimizer"]
criterion = experiment_setup["criterion"]
config = experiment_setup["cfg"]

In [74]:
trafficSignClassifier = TrafficSignClassifier(config, model, criterion, optimizer)

## Training

In [75]:
# Callbacks

checkpoint_callback = ModelCheckpoint(
    monitor="val_acc",
    mode="max", # max for accuracy, min for loss
    dirpath=constants.MODEL_CHECKPOINTS_DIR,
    filename=f"{cfg_name}-{{epoch:02d}}-{{val_acc:.3f}}",
    save_top_k=1,
)

early_stop_callback = EarlyStopping(
    monitor="val_acc",
    mode="max",
    patience=15,
    verbose=True,
)

In [76]:
# Logger

logger = TensorBoardLogger(
    save_dir=constants.LOGS_DIR,
    name=cfg_name
)


In [77]:
# Trainer

trainer = pl.Trainer(
    max_epochs=100,
    callbacks=[checkpoint_callback, early_stop_callback],
    accelerator="auto",
    check_val_every_n_epoch=1,
    logger=logger,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [78]:
trainer.fit(trafficSignClassifier, train_loader, val_loader)

F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory F:\Software\Dataspell\TrafficSigns\Experiments\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | EfficientNet     | 4.1 M  | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
4.1 M     Trainable params
864       Non-trainable params
4.1 M     Total params
16.251    Total estimated model params size (MB)
338       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_acc did not improve in the last 15 records. Best score: 0.995. Signaling Trainer to stop.
